# Dynamo Models of the Solar Cycle — Implementation / 구현

**Paper**: Charbonneau, P. (2010). *Dynamo Models of the Solar Cycle*. Living Reviews in Solar Physics, 7, 3. DOI: 10.12942/lrsp-2010-3

이 노트북은 Charbonneau (2010) 리뷰의 핵심 다이나모 모델들을 선택적으로 구현합니다. 리뷰 논문이므로 전체를 구현할 수는 없고, 각 모델 계열의 **가장 단순한 버전**을 NumPy로 직접 구현하여 수학과 물리를 체험합니다.

This notebook selectively implements the key dynamo models reviewed in Charbonneau (2010). Since it is a review, we cannot reproduce everything — instead we implement the **minimal working version** of each model family using only NumPy, to build intuition for the mathematics and physics.

**Contents / 구성:**
1. MHD induction equation: kinematic advection-diffusion / MHD 유도 방정식
2. Internal differential rotation profile (helioseismology fit) / 내부 차등 회전 프로파일
3. 1D αΩ dynamo wave — Parker's dispersion relation / αΩ 다이나모 파동
4. Kinematic αΩ dynamo in 2D meridional plane / 2D 자오면 αΩ 다이나모
5. α-quenching nonlinear saturation / α-담금질 비선형 포화
6. Flux-transport dynamo: meridional circulation & cycle period / 플럭스 수송 다이나모
7. Stochastic α & grand minima / 확률적 α와 grand minima

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

RNG = np.random.default_rng(42)
R_SUN = 6.96e8  # m
YEAR = 3.154e7  # s

## Part 1: MHD Induction Equation — Kinematic Toy Problem / MHD 유도 방정식 — 운동학 장난감 문제

$$
\frac{\partial \mathbf{B}}{\partial t} = \nabla\times(\mathbf{u}\times\mathbf{B}) - \nabla\times(\eta\,\nabla\times\mathbf{B})
$$

1차원 단순화된 경우, $B = B(x,t)$, $u = U$ (상수) 에서 위 방정식은 advection-diffusion 방정식으로 줄어듭니다:

$$
\frac{\partial B}{\partial t} + U\frac{\partial B}{\partial x} = \eta\frac{\partial^2 B}{\partial x^2}
$$

자기 Reynolds 수 $R_m = UL/\eta$가 클수록 **이류가 지배**, 작을수록 **확산이 지배**합니다. 이 toy problem은 다이나모가 **유도와 확산의 경쟁**이라는 본질을 보여줍니다.

In the 1D case with $B=B(x,t)$ and constant $u=U$, the induction equation reduces to an advection-diffusion equation. Large $R_m=UL/\eta$ → **advection dominates**; small $R_m$ → **diffusion dominates**. This toy problem illustrates the essential dynamo balance.

In [ ]:
def evolve_advection_diffusion(
    B0: np.ndarray,
    U: float,
    eta: float,
    dx: float,
    dt: float,
    n_steps: int,
) -> np.ndarray:
    """Evolve 1D advection-diffusion with periodic BCs using upwind + centered diffusion.

    Args:
        B0: Initial field array of shape (N,).
        U: Advection velocity (m/s).
        eta: Magnetic diffusivity (m^2/s).
        dx: Grid spacing (m).
        dt: Time step (s).
        n_steps: Number of time steps to integrate.

    Returns:
        History array of shape (n_steps+1, N) of the magnetic field.
    """
    B = B0.copy()
    history = [B.copy()]
    for _ in range(n_steps):
        B_plus = np.roll(B, -1)
        B_minus = np.roll(B, 1)
        if U >= 0:
            adv = -U * (B - B_minus) / dx
        else:
            adv = -U * (B_plus - B) / dx
        diff = eta * (B_plus - 2 * B + B_minus) / dx**2
        B = B + dt * (adv + diff)
        history.append(B.copy())
    return np.array(history)


N = 200
L = 1.0
dx = L / N
x = np.linspace(0, L, N, endpoint=False)
B0 = np.exp(-((x - 0.3) / 0.05) ** 2)

U = 0.5
cases = {'$R_m=1$': U * L / 1.0, '$R_m=10$': U * L / 0.05, '$R_m=100$': U * L / 0.005}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (label, Rm) in zip(axes, cases.items()):
    eta = U * L / Rm
    dt = 0.5 * min(dx / abs(U), 0.5 * dx**2 / eta)
    n_steps = int(0.8 / dt)
    hist = evolve_advection_diffusion(B0, U, eta, dx, dt, n_steps)
    ax.plot(x, B0, 'k--', label='initial')
    for frac in [0.25, 0.5, 0.75, 1.0]:
        idx = int(frac * n_steps)
        ax.plot(x, hist[idx], label=f't={frac*0.8:.2f}')
    ax.set_title(label)
    ax.set_xlabel('x')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)
axes[0].set_ylabel('B(x,t)')
plt.suptitle('Advection-diffusion: high $R_m$ preserves structure, low $R_m$ smears it')
plt.tight_layout()
plt.show()

## Part 2: Internal Differential Rotation Profile / 내부 차등 회전 프로파일

Helioseismology (§2의 Figure 4)는 태양의 내부 회전을 관측했습니다. Charbonneau가 인용한 대표적 파라메트릭 피팅:

$$
\Omega(r,\theta) = \Omega_c + \frac{1}{2}\left[1 + \mathrm{erf}\!\left(\frac{r - r_c}{w}\right)\right]\bigl[\Omega_s(\theta) - \Omega_c\bigr]
$$

$$
\Omega_s(\theta) = \Omega_{\rm eq} - a_2\cos^2\theta - a_4\cos^4\theta
$$

파라미터 / Parameters: $\Omega_c = 2\pi\times 432.8$ nHz, $\Omega_{\rm eq} = 2\pi\times 460.7$ nHz, $a_2 = 62.69$, $a_4 = 67.13$ (all in nHz), $r_c = 0.7 R_\odot$, $w = 0.05 R_\odot$.

The tachocline (~0.7 $R_\odot$) is where **radial shear** $\partial\Omega/\partial r$ is strongest — the home of the Ω-effect.

tachocline(~0.7 $R_\odot$)에서 반경 방향 전단 $\partial\Omega/\partial r$가 가장 강함 — Ω-효과의 주 무대.

In [ ]:
def differential_rotation(r: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """Return angular velocity Omega(r, theta) in nHz from a standard helioseismic fit.

    Args:
        r: Radial coordinate in units of R_sun, shape (Nr,) or (Nr, Ntheta).
        theta: Colatitude in radians, shape (Ntheta,) or (Nr, Ntheta).

    Returns:
        Omega array in nHz, broadcast over r and theta.
    """
    from scipy.special import erf

    omega_c = 432.8
    omega_eq = 460.7
    a2, a4 = 62.69, 67.13
    r_c, w = 0.7, 0.05
    R, T = np.meshgrid(r, theta, indexing='ij')
    omega_s = omega_eq - a2 * np.cos(T) ** 2 - a4 * np.cos(T) ** 4
    mix = 0.5 * (1 + erf((R - r_c) / w))
    return omega_c + mix * (omega_s - omega_c)


r = np.linspace(0.5, 1.0, 120)
theta = np.linspace(0.01, np.pi - 0.01, 120)
Omega = differential_rotation(r, theta)

R, T = np.meshgrid(r, theta, indexing='ij')
X = R * np.sin(T)
Z = R * np.cos(T)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
pc = axes[0].contourf(X, Z, Omega, levels=20, cmap='plasma')
axes[0].set_aspect('equal')
axes[0].set_title('Differential rotation $\\Omega(r,\\theta)$ [nHz]')
axes[0].set_xlabel('$\\varpi/R_\\odot$')
axes[0].set_ylabel('$z/R_\\odot$')
plt.colorbar(pc, ax=axes[0])

for lat_deg in [0, 30, 45, 60, 75]:
    tlat = np.pi / 2 - np.deg2rad(lat_deg)
    idx = np.argmin(np.abs(theta - tlat))
    axes[1].plot(r, Omega[:, idx], label=f'{lat_deg}°')
axes[1].axvline(0.7, color='k', ls=':', alpha=0.5, label='tachocline')
axes[1].set_xlabel('$r/R_\\odot$')
axes[1].set_ylabel('$\\Omega$ [nHz]')
axes[1].set_title('Radial cuts — strong shear at $r=0.7R_\\odot$')
axes[1].legend(title='latitude')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: 1D αΩ Dynamo Wave — Parker's Dispersion Relation / αΩ 다이나모 파동

Parker (1955) showed that the linearized αΩ equations support a **traveling-wave solution**. Plane-wave ansatz $A, B \propto \exp(i\mathbf{k}\cdot\mathbf{x} - i\omega t)$ gives the dispersion relation:

$$
(\sigma + \eta_T k^2)^2 = i\,\alpha\,\frac{\partial\Omega}{\partial r}\,k
$$

with $\sigma = -i\omega$. Extracting real and imaginary parts gives

$$
\sigma_{\rm growth} = -\eta_T k^2 + \sqrt{\tfrac{1}{2}|\alpha\,\partial_r\Omega\,k|}, \qquad \omega = \text{sign}(\alpha\,\partial_r\Omega)\sqrt{\tfrac{1}{2}|\alpha\,\partial_r\Omega\,k|}
$$

**Parker-Yoshimura rule**: 파동의 위상 속도는 $\alpha\,\partial_r\Omega > 0$이면 극방향, $< 0$이면 적도방향. 태양의 적도 butterfly를 맞추려면 음의 부호가 필요하다.

The **Parker-Yoshimura rule** says waves propagate poleward when $\alpha\,\partial_r\Omega > 0$ and equatorward when $< 0$. Matching the observed equatorward butterfly requires the negative sign.

In [ ]:
def parker_wave_rates(k: np.ndarray, alpha: float, dOmega_dr: float, eta_T: float):
    """Compute growth rate and frequency of Parker's dynamo wave.

    Args:
        k: Wavenumber array (1/m).
        alpha: α-effect amplitude (m/s).
        dOmega_dr: Radial shear (1/s/m).
        eta_T: Turbulent magnetic diffusivity (m^2/s).

    Returns:
        Tuple (sigma, omega): growth rate and frequency (1/s).
    """
    D = alpha * dOmega_dr
    amp = np.sqrt(0.5 * np.abs(D * k))
    sigma = -eta_T * k**2 + amp
    omega = np.sign(D) * amp
    return sigma, omega


alpha = 1.0
eta_T = 1.0e8
dOmega_dr_pos = +3.0e-15
dOmega_dr_neg = -3.0e-15
k = np.linspace(1e-9, 2e-7, 300)

sigma_pos, omega_pos = parker_wave_rates(k, alpha, dOmega_dr_pos, eta_T)
sigma_neg, omega_neg = parker_wave_rates(k, alpha, dOmega_dr_neg, eta_T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k, sigma_pos, label=r'$\alpha\partial_r\Omega > 0$ (poleward)')
axes[0].plot(k, sigma_neg, '--', label=r'$\alpha\partial_r\Omega < 0$ (equatorward)')
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_xlabel('k (1/m)')
axes[0].set_ylabel('growth rate $\\sigma$ (1/s)')
axes[0].set_title('Growth rate vs wavenumber')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(k, omega_pos)
axes[1].plot(k, omega_neg, '--')
axes[1].set_xlabel('k (1/m)')
axes[1].set_ylabel('frequency $\\omega$ (1/s)')
axes[1].set_title('Parker-Yoshimura rule: sign of $\\omega$ = direction')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

k_opt = np.sqrt(np.abs(alpha * dOmega_dr_neg) / (8 * eta_T**2)) ** (1 / 3)
T_wave = 2 * np.pi / np.abs(parker_wave_rates(np.array([k_opt]), alpha, dOmega_dr_neg, eta_T)[1][0])
print(f'Optimal k ≈ {k_opt:.2e} 1/m, period ≈ {T_wave/YEAR:.1f} yr')

## Part 4: 2D Kinematic αΩ Dynamo in a Meridional Plane / 2D 자오면 αΩ 다이나모

We solve the axisymmetric mean-field equations in the $(r,\theta)$ plane:

$$
\frac{\partial A}{\partial t} = \eta_T\!\left(\nabla^2 - \frac{1}{\varpi^2}\right)A + \alpha B
$$

$$
\frac{\partial B}{\partial t} = \eta_T\!\left(\nabla^2 - \frac{1}{\varpi^2}\right)B + \varpi\,\mathbf{B}_p\cdot\nabla\Omega
$$

with $\varpi = r\sin\theta$, $\mathbf{B}_p = \nabla\times(A\hat\phi)$. We use:
- α(r,θ) concentrated just above the tachocline (classical αΩ) with antisymmetric latitude profile
- Ω(r,θ) from Part 2
- Explicit time stepping on a uniform grid

tachocline 바로 위에 위치한 α(r,θ)와 Part 2의 Ω(r,θ)를 써서 축대칭 mean-field αΩ 다이나모를 명시적 시간 적분으로 풉니다. Butterfly diagram을 재현합니다.

**Note**: This is a **minimal pedagogical solver** (not production). The radial domain is truncated to $r \in [0.6, 1.0]R_\odot$ with simple BCs. Quantitative realism is secondary to showing the mechanism.

In [ ]:
def run_alphaomega_dynamo(
    nr: int = 40,
    nth: int = 60,
    n_steps: int = 6000,
    C_alpha: float = -7.5,
    C_Omega: float = 5e4,
    eta_T: float = 5e-4,
    dt: float = 1e-4,
) -> dict:
    """Run a kinematic αΩ mean-field dynamo in a meridional plane (dimensionless units).

    Uses the full dynamo-number formulation: time is in units of the diffusion time
    τ_η = R_sun^2 / η_T. The dynamo number D = C_α · C_Ω sets the operating regime.

    Args:
        nr: Number of radial grid points.
        nth: Number of colatitude grid points.
        n_steps: Number of time steps.
        C_alpha: Dimensionless α-effect amplitude (sign matters).
        C_Omega: Dimensionless Ω-effect amplitude.
        eta_T: Dimensionless diffusivity (for numerical stability).
        dt: Time step (dimensionless).

    Returns:
        Dict with time series, butterfly slice, and final 2D fields.
    """
    r = np.linspace(0.6, 1.0, nr)
    theta = np.linspace(0.05, np.pi - 0.05, nth)
    dr = r[1] - r[0]
    dth = theta[1] - theta[0]
    R, T = np.meshgrid(r, theta, indexing='ij')
    varpi = R * np.sin(T)

    Omega = differential_rotation(r, theta)
    Omega = (Omega - Omega.mean()) / Omega.ptp()

    alpha = np.cos(T) * np.exp(-((R - 0.72) / 0.05) ** 2)

    A = 1e-3 * np.sin(T) * np.cos(np.pi * (R - 0.6) / 0.4)
    B = np.zeros_like(A)

    def lap(F):
        F_r = np.zeros_like(F)
        F_th = np.zeros_like(F)
        F_r[1:-1, :] = (F[2:, :] - 2 * F[1:-1, :] + F[:-2, :]) / dr**2
        F_r[1:-1, :] += (F[2:, :] - F[:-2, :]) / (R[1:-1, :] * dr)
        F_th[:, 1:-1] = (F[:, 2:] - 2 * F[:, 1:-1] + F[:, :-2]) / (R[:, 1:-1] ** 2 * dth**2)
        F_th[:, 1:-1] += (np.cos(T[:, 1:-1]) / np.sin(T[:, 1:-1])) * (
            F[:, 2:] - F[:, :-2]
        ) / (2 * R[:, 1:-1] ** 2 * dth)
        return F_r + F_th

    def grad_r(F):
        G = np.zeros_like(F)
        G[1:-1, :] = (F[2:, :] - F[:-2, :]) / (2 * dr)
        return G

    def grad_th(F):
        G = np.zeros_like(F)
        G[:, 1:-1] = (F[:, 2:] - F[:, :-2]) / (2 * dth)
        return G

    idx_surface = nr - 2
    butterfly = np.zeros((n_steps // 10, nth))
    polar_timeseries = np.zeros(n_steps // 10)
    times = np.zeros(n_steps // 10)

    for step in range(n_steps):
        L_A = lap(A) - A / np.maximum(varpi**2, 1e-6)
        L_B = lap(B) - B / np.maximum(varpi**2, 1e-6)

        Bp_r = grad_th(A) / np.maximum(R * np.sin(T), 1e-6) + A * np.cos(T) / np.maximum(R * np.sin(T), 1e-6)
        Bp_th = -grad_r(A) - A / R

        shear_term = varpi * (Bp_r * grad_r(Omega) + Bp_th * grad_th(Omega) / R)

        dAdt = eta_T * L_A + C_alpha * alpha * B
        dBdt = eta_T * L_B + C_Omega * shear_term

        A = A + dt * dAdt
        B = B + dt * dBdt

        A[0, :] = 0
        A[-1, :] = 0
        B[0, :] = 0
        B[-1, :] = 0
        A[:, 0] = 0
        A[:, -1] = 0
        B[:, 0] = 0
        B[:, -1] = 0

        max_abs = np.max(np.abs(B))
        if max_abs > 1e6:
            A /= max_abs
            B /= max_abs

        if step % 10 == 0:
            i = step // 10
            butterfly[i, :] = B[idx_surface - 10, :]
            polar_timeseries[i] = A[idx_surface, 3] - A[idx_surface, -3]
            times[i] = step * dt

    return {
        'A': A,
        'B': B,
        'r': r,
        'theta': theta,
        'alpha': alpha,
        'Omega': Omega,
        'butterfly': butterfly,
        'polar': polar_timeseries,
        't': times,
    }


out = run_alphaomega_dynamo(C_alpha=-12.0, C_Omega=3e4, eta_T=8e-4, dt=5e-5, n_steps=4000)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

latitudes = 90 - np.rad2deg(out['theta'])
bf = out['butterfly']
bf_norm = bf / (np.max(np.abs(bf)) + 1e-12)
im = axes[0, 0].pcolormesh(
    out['t'], latitudes, bf_norm.T, cmap='RdBu_r', shading='auto',
    norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1),
)
axes[0, 0].set_xlabel('time (diffusion units)')
axes[0, 0].set_ylabel('latitude (deg)')
axes[0, 0].set_title('Butterfly diagram — toroidal field at sub-surface')
plt.colorbar(im, ax=axes[0, 0])

axes[0, 1].plot(out['t'], out['polar'])
axes[0, 1].set_xlabel('time (diffusion units)')
axes[0, 1].set_ylabel('N-S polar asymmetry of A')
axes[0, 1].set_title('Polar field time series')
axes[0, 1].grid(alpha=0.3)

R2, T2 = np.meshgrid(out['r'], out['theta'], indexing='ij')
X2 = R2 * np.sin(T2)
Z2 = R2 * np.cos(T2)
pc = axes[1, 0].contourf(X2, Z2, out['B'], levels=15, cmap='RdBu_r')
axes[1, 0].set_aspect('equal')
axes[1, 0].set_title('Toroidal field $B_\\phi$ (final snapshot)')
axes[1, 0].set_xlabel('$\\varpi/R_\\odot$')
axes[1, 0].set_ylabel('$z/R_\\odot$')
plt.colorbar(pc, ax=axes[1, 0])

pc2 = axes[1, 1].contourf(X2, Z2, out['A'], levels=15, cmap='RdBu_r')
axes[1, 1].set_aspect('equal')
axes[1, 1].set_title('Poloidal potential $A$ (final snapshot)')
axes[1, 1].set_xlabel('$\\varpi/R_\\odot$')
axes[1, 1].set_ylabel('$z/R_\\odot$')
plt.colorbar(pc2, ax=axes[1, 1])

plt.tight_layout()
plt.show()
print('Toroidal field amplitude (final):', np.max(np.abs(out['B'])))
print('Poloidal field amplitude (final):', np.max(np.abs(out['A'])))

## Part 5: α-quenching Nonlinear Saturation / α-담금질 비선형 포화

The simplest amplitude-limiter: reduce α when $|B|$ grows large.

$$
\alpha(B) = \frac{\alpha_0}{1 + (B/B_{\rm eq})^2}
$$

This saturates the dynamo and produces limit-cycle oscillations. We integrate the reduced **0D amplitude equations**:

$$
\dot A = -\lambda A + \frac{\alpha_0}{1+(B/B_{\rm eq})^2} B, \qquad \dot B = -\lambda B + \omega_\Omega A
$$

가장 단순한 진폭 제한: $|B|$가 커지면 α가 감소. 0D 진폭 방정식을 적분하여 limit cycle을 확인.

This toy system is a nonlinear oscillator — qualitatively it reproduces the bounded periodic cycle of the solar dynamo.

In [ ]:
def alphaquench_dynamo_0d(
    alpha0: float,
    omega_omega: float,
    lam: float,
    B_eq: float,
    T_end: float,
    dt: float,
    A0: float = 1e-3,
    B0: float = 0.0,
) -> np.ndarray:
    """Integrate a 0D α-quenched αΩ amplitude system with RK4.

    Args:
        alpha0: Linear α amplitude.
        omega_omega: Ω-effect rate (1/s).
        lam: Diffusive damping rate (1/s).
        B_eq: Equipartition toroidal field.
        T_end: Integration end time.
        dt: Time step.
        A0: Initial poloidal amplitude.
        B0: Initial toroidal amplitude.

    Returns:
        Array of shape (n_steps+1, 3): columns are [t, A, B].
    """

    def rhs(A, B):
        alpha_eff = alpha0 / (1 + (B / B_eq) ** 2)
        return -lam * A + alpha_eff * B, -lam * B + omega_omega * A

    n = int(T_end / dt)
    out = np.zeros((n + 1, 3))
    A, B = A0, B0
    for i in range(n + 1):
        out[i] = (i * dt, A, B)
        k1A, k1B = rhs(A, B)
        k2A, k2B = rhs(A + 0.5 * dt * k1A, B + 0.5 * dt * k1B)
        k3A, k3B = rhs(A + 0.5 * dt * k2A, B + 0.5 * dt * k2B)
        k4A, k4B = rhs(A + dt * k3A, B + dt * k3B)
        A += dt * (k1A + 2 * k2A + 2 * k3A + k4A) / 6
        B += dt * (k1B + 2 * k2B + 2 * k3B + k4B) / 6
    return out


sol = alphaquench_dynamo_0d(
    alpha0=0.5, omega_omega=2.0, lam=0.05, B_eq=1.0, T_end=120.0, dt=0.01,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sol[:, 0], sol[:, 1], label='poloidal A')
axes[0].plot(sol[:, 0], sol[:, 2], label='toroidal B')
axes[0].set_xlabel('time')
axes[0].set_ylabel('amplitude')
axes[0].set_title('α-quenched dynamo time series')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(sol[:, 1], sol[:, 2])
axes[1].set_xlabel('A (poloidal)')
axes[1].set_ylabel('B (toroidal)')
axes[1].set_title('Phase portrait — limit cycle')
axes[1].grid(alpha=0.3)
axes[1].set_aspect('equal', 'box')
plt.tight_layout()
plt.show()

## Part 6: Flux-transport Dynamo — Cycle Period from Meridional Circulation / 자오선 순환으로부터 주기

The central prediction of the flux-transport dynamo (§4.4) is that the cycle period is set by the **travel time of meridional circulation**, not the diffusion time:

$$
T_{\rm cycle} \approx 2\,\frac{L_{\rm mer}}{u_{\rm mer}}
$$

For the Sun, $L_{\rm mer} \sim R_\odot$ and $u_{\rm mer} \sim 20$ m/s give ~11 yr. Let's verify with a schematic conveyor-belt advection simulation.

flux-transport dynamo(§4.4)의 핵심 예측: 주기는 확산 시간이 아닌 **자오선 순환 일주 시간**으로 결정됨. ~20 m/s가 ~11년 주기를 자연스럽게 준다.

In [ ]:
def conveyor_cycle_period(u_mer_list: np.ndarray, L_mer: float = R_SUN) -> np.ndarray:
    """Compute flux-transport dynamo cycle period from meridional speed.

    Args:
        u_mer_list: Array of meridional circulation speeds (m/s).
        L_mer: Half-circumference along the conveyor belt (m).

    Returns:
        Array of cycle periods in years.
    """
    return 2.0 * L_mer / u_mer_list / YEAR


u_mer_range = np.linspace(5, 40, 40)
periods = conveyor_cycle_period(u_mer_range)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(u_mer_range, periods, lw=2)
ax.axhline(11, color='r', ls='--', label='observed Schwabe = 11 yr')
ax.axvspan(15, 25, color='gold', alpha=0.3, label='observed u_mer = 15–25 m/s')
ax.set_xlabel('meridional speed $u_{\\rm mer}$ (m/s)')
ax.set_ylabel('predicted cycle period (yr)')
ax.set_title('Flux-transport dynamo: period vs. circulation speed')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'At u_mer = 20 m/s: period ≈ {conveyor_cycle_period(np.array([20.0]))[0]:.1f} yr')

## Part 7: Stochastic α and Grand Minima / 확률적 α와 grand minima

We extend Part 5 by adding multiplicative noise to α to mimic **discrete BMR-emergence stochasticity** (§6). Sufficiently large noise can drive the system below critical and produce Maunder-like temporary shutdowns.

$$
\alpha_{\rm eff}(t) = \frac{\alpha_0[1 + \sigma_\alpha\,\xi(t)]}{1 + (B/B_{\rm eq})^2}, \qquad \xi(t)\sim\mathcal{N}(0,1)
$$

Part 5의 모델에 α의 곱셈적 노이즈를 추가 — BMR emergence의 이산적 확률성을 흉내. 충분히 큰 노이즈가 Maunder-like 일시 정지 상태를 유발.

In [ ]:
def stochastic_alphaquench(
    alpha0: float,
    omega_omega: float,
    lam: float,
    B_eq: float,
    sigma_alpha: float,
    T_end: float,
    dt: float,
) -> np.ndarray:
    """Integrate 0D α-quenched dynamo with multiplicative α-noise (Euler-Maruyama).

    Args:
        alpha0: Mean α amplitude.
        omega_omega: Ω-effect rate.
        lam: Damping rate.
        B_eq: Equipartition field.
        sigma_alpha: Relative noise amplitude.
        T_end: End time.
        dt: Time step.

    Returns:
        Array of shape (n_steps+1, 3): columns [t, A, B].
    """
    n = int(T_end / dt)
    out = np.zeros((n + 1, 3))
    A, B = 1e-3, 0.0
    for i in range(n + 1):
        out[i] = (i * dt, A, B)
        noise = RNG.normal(0.0, 1.0)
        alpha_eff = alpha0 * (1 + sigma_alpha * noise) / (1 + (B / B_eq) ** 2)
        A_next = A + dt * (-lam * A + alpha_eff * B)
        B_next = B + dt * (-lam * B + omega_omega * A)
        A, B = A_next, B_next
    return out


T_end = 1200.0
dt = 0.02

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, sigma in zip(axes, [0.0, 0.4, 0.9]):
    sol = stochastic_alphaquench(
        alpha0=0.3, omega_omega=2.0, lam=0.05, B_eq=1.0,
        sigma_alpha=sigma, T_end=T_end, dt=dt,
    )
    ax.plot(sol[:, 0], np.abs(sol[:, 2]))
    ax.set_ylabel('|B| (amplitude)')
    ax.set_title(f'α-noise $\\sigma_\\alpha$ = {sigma}')
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('time (arbitrary units)')
plt.suptitle('Stochastic α → intermittency resembling grand minima')
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | In Charbonneau 2010 | Our implementation | Modern tools / 현대 도구 |
|---|---|---|---|
| MHD induction balance | §2, Eq. (1) | Part 1: 1D advection-diffusion | Pencil Code, Dedalus |
| Differential rotation profile | §2, Fig. 4 | Part 2: erf-based fit | GONG/SDO-HMI helioseismology |
| Parker-Yoshimura rule | §4.2, Eq. (55) | Part 3: analytical dispersion | — |
| αΩ kinematic dynamo | §4.2 | Part 4: 2D finite-difference | STELLA, MURaM, EULAG-MHD |
| α-quenching | §5 | Part 5: 0D ODE limit cycle | Same form in all FTD codes |
| Flux-transport dynamo | §4.4 | Part 6: schematic period law | Surya, Dikpati-Gilman |
| Grand minima / stochastic α | §6 | Part 7: multiplicative noise | Cameron & Schüssler (2017) |

**Key lessons from implementation / 구현에서 얻은 핵심 교훈:**

1. **The induction equation is both generator and graveyard.** High $R_m$ preserves structure; low $R_m$ smears everything — every model lives on this balance. / 유도 방정식은 생성자이자 무덤. 모든 모델은 $R_m$ 균형 위에 있다.
2. **The tachocline emerges from a parametric fit, but its physical role is universal.** Ω-shear there drives toroidal generation in all modern models. / tachocline은 피팅 산물이지만 물리적 역할은 보편.
3. **The Parker-Yoshimura sign rule is a hard constraint.** Any 1D analytical check can decide if an αΩ model will butterfly equatorward. / Parker-Yoshimura 부호 규칙은 엄격한 제약.
4. **2D αΩ solvers are delicate.** Our toy code shows that butterfly patterns depend strongly on the spatial profile of α — not just its sign. / 2D 솔버는 민감하다.
5. **Nonlinear saturation needs little physics.** Even a simple algebraic α-quenching already gives bounded limit cycles. / 비선형 포화에는 약간의 물리로 충분.
6. **Cycle period ~11 yr emerges naturally from observed meridional speed.** This single scaling explains why FTDs dominate modern thinking. / 11년 주기는 자오선 순환 속도로부터 자연스럽다.
7. **Stochasticity in α is sufficient to produce Maunder-like minima.** Not conclusive, but plausible — consistent with §6. / α 확률성으로도 grand minima-like 거동이 나타난다.

**Connections to next papers / 다음 논문과의 연결:**

- **LRSP #15 Fan (2021)** — provides the microscopic physics (buoyant flux tube rise) underlying BL α.
- **LRSP (Hathaway 2015)** — details the observations our models must match.
- **Living Reviews on Global MHD Simulations** — what replaces the mean-field approach at large Reynolds numbers.